# Νευρωνικά Δίκτυα για Ταξινόμηση

Σε αυτό το Notebook θα δούμε πως προσαρμόζουμε τα νευρωνικά δίκτυα που έχουμε ήδη δεί, για να ταιριάζουν με την ταξινόμηση. Συγκεκριμένα, θα καλύψουμε:
1. Softmax -- νευρωνικά για ταξινόμηση
2. Πως χρησιμοποιηούνται στην ταξινόμηση δεδομένων. Θα δούμε ένα πολύ απλό δισδιάστατο παράδειγμα.
2. Ισοϋψείς καμπύλες

```
Κωνσταντίνος Καραμανής: constantine@utexas.edu
http://users.ece.utexas.edu/~cmcaram/
The University of Texas at Austin
Archimedes/Athena RC
```

# Ταξινόμηση και SoftMax

Εώς τώρα, να νευρωνικά δίκτυα που έχουμε φτιάξει παίρνουν τα δεδομένα εισόδου σαν input, και σαν output παράγουν έναν αριθμό.

Αυτό το πρότυπο ταιριάζει με την παλινδρόμηση. Πρέπει να το προσαρμόσουμε για να ταιριάζει με την ταξινόμηση. Αντί να απαιτήσουμε να μας πεί το νευρωνικό δίκτυο σε ποιά κατηγορία ανήκει κάθε δεδομένο εισόδου, θα του επιτρέψουμε να μας πεί που *νομίζει* πως ανήκει.

Για παράδειγμα, σε πρόβλημα ταξινόμησης με δύο κατηγορίες, αντί να βγεί αποτέλεσμα $(1,0)$ (κατηγορία 1) ή $(0,1)$ (κατηγορία 2), θα μπορούσε να βγάλει αποτέλεσμα $(0,85, 0,15)$. Αυτό το ερμηνεύουμε ώς ένδειξε πως το νευρωνικό δίκτυο πιστεύει πως *μάλλον* ανήκει στην πρώτη κατηγορία (85%), αλλά μπορεί (15%) να ανήκει στην δεύτερη κατηγορία.

Εξηγούμε την ιδέα με ένα απλό παράδειγμα:
```
class SimpleClassifier(nn.Module):
    def __init__(self):
        super(SimpleClassifier, self).__init__()
        self.fc = nn.Linear(2, 2)

    def forward(self, x):
        x = self.fc(x)
```
Τις έχουμε δεί όλες αυτές τις εντολές παραπάνω. Αυτό το νευρωνικό δίκτυο έχει 6 παραμέτρους: α, β, γ, δ, c1, c2, και με input $(x_1,x_2)$ το output είναι:
$$
(z_1,z_2) = (α x_1 + β x_2 + c_1, γ x_1 + δ x_2 + c_2).
$$


Το softmax είναι βασικά ένα normalization, που παίρνει τους δύο αριθμούς $(z_1,z_2)$ και επιστρέφει αριθμούς $(p_1,p_2)$ μεταξύ 0 και 1, που επίσης ικανοποιούν $p_1 + p_2 = 1$.

softmax$(z_1,z_2)=(p_1,p_2)$, όπου

$$
p_1 = \frac{exp(z_1)}{(exp(z_1) + exp(z_2))}
$$


$$
p_2 = \frac{exp(z_2)}{(exp(z_1) + exp(z_2))}.
$$
Παρατηρούμε πως $0 \leq p_1, p_2 \leq 1$, και επίσης $p_1 + p_2 = 1$.

Ο ολοκληρωμένος κώδικας απλά προσθέτει το ``softmax`` με μία πρόσθετη εντολή: ``F.softmax(x, dim=1)``
```
class SimpleClassifier(nn.Module):
    def __init__(self):
        super(SimpleClassifier, self).__init__()
        self.fc = nn.Linear(2, 2)  # Input dimension is 2, output dimension is 2 (one for each class)

    def forward(self, x):
        x = self.fc(x)
        return F.softmax(x, dim=1)
```

### Softmax για παραπάνω από 2 κατηγορίες

Εφαρμόζουμε την ίδια ιδέα και για ταξινόμηση με παραπάνω από 2 κατηγορίες. Το CIFAR-10 έχει 10. Το πιο απλό νευρωνικό δίκτυο που θα μπορούσαμε να προσπαθήσουμε να χρησιμοποιήσουμε για το CIFAR-10 θα είχε την μορφή:
```
class CIFARSimpleNet(nn.Module):
    def __init__(self):
        super(CIFARSimpleNet, self).__init__()
        self.fc = nn.Linear(3072, 10)

    def forward(self, x):
        x = self.fc(x)
        return F.softmax(x, dim=1)
```
Το πρώτο επίπεδο παίρνει τις 32x32x3 = 3.072 input-features, και παράγει 10 αριθμούς, ας τους πούμε $z_1, z_2, \dots, z_{10}$.

Μετά το softmax κάνει το εξής:

$$
p_1 = \frac{exp(z_1)}{(exp(z_1) + exp(z_2) + \cdots +exp(z_{10})}
$$


$$
p_2 = \frac{exp(z_2)}{(exp(z_1) + exp(z_2) + \cdots +exp(z_{10})}
$$

$$
\vdots
$$

$$
p_{10} = \frac{exp(z_{10})}{(exp(z_1) + exp(z_2) + \cdots +exp(z_{10})}
$$

Παρατηρούμε πως όλα τα $p_i$ ικανοποιούν: $0 \leq p_i \leq 1$, και $p_1+ \cdots + p_{10} = 1$, οπότε μπορούμε να τα θεωρήσουμε ως την πιθανότητα με την οποία το μοντέλο μας "πιστεύει" πως το ${\bf x}$ ανήκει σε κάθε κατηγορία.



## **Άσκηση**: softmax

Γράψτε μία συνάρτηση που υπολογίζει το ``softmax``.

```
def soft_max(z):
    XXXX
    ΧΧΧΧ
    ΧΧΧΧ
    return p
```
Εδώ το $z$ είναι ``array`` από $n$ αριθμούς: $z = (z_1,...,z_n)$, και παρομοίως, το $p$ που επιστρέφει η συνάρτηση, είναι ``array`` με $n$ αριθμούς $p =(p_1,...,p_n)$ μεταξύ του μηδενός και της μονάδας ($0 \leq p_i \leq 1$), και που αθροίζουν σε 1: $p_1 + \cdots p_n = 1$.

# Και τώρα στην πράξη

1. Θα δημιουργήσουμε δεδομένα τεχνητά για ένα απλό πρόβλημα ταξινόμησης,
2. Θα φτιάξουμε μια οικογένεια νευρωνικών δικτύων που αντιστοιχεί στην παραπάνω συζήτηση,
3. Θα επιλέξουμε ένα από αυτά, διαλέγοντας τιμές για τις παραμέτρους του δικτύου (όπως κάναμε και στο προηγούμενο Notebook),
4. Θα δούμε πώς τα πάει στο πρόβλημά μας, χρησιμοποιόντας **ισοϋψείς καμπύλες**.

In [ ]:
# import the Pytorch library
import torch
import torch.nn as nn
import torch.nn.functional as F


### Δημιουργούμε δεδομένα

Δημιουργούνε ένα απλό πρόβλημα ταξινόμησης με τεχνητά δεδομένα χωρισμένα σε δύο κατηγορίες.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Seed for reproducibility
np.random.seed(42)

# Generate positive examples in the upper right quadrant
X_positive = np.random.uniform(0.5,1.0,(20,2))

# Generate negative examples in the lower left quadrant
X_negative = np.random.uniform(0.0, 0.5,(20,2))

# Plotting the data
plt.figure(figsize=(5, 5))
plt.scatter(X_positive[:,0],X_positive[:,1], marker='^', color='blue', label='Positive (Triangle)')
plt.scatter(X_negative[:,0],X_negative[:,1], marker='o', color='red', label='Negative (Circle)')
plt.title('Simple Classification Dataset')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True)
plt.show()


### Ορίζουμε την οικογένεια των νευρωνικών

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleClassifier(nn.Module):
    def __init__(self):
        super(SimpleClassifier, self).__init__()
        self.fc = nn.Linear(2, 2)  # Input dimension is 2, output dimension is 2 (one for each class)

    def forward(self, x):
        x = self.fc(x)
        return F.softmax(x, dim=1)


## Διαλέγουμε Ένα Νευρωνικό Δίκτυο Από την Οικογένεια

Από την οικογένεια που ορίσαμε, επιλέγουμε ένα συγκεκριμένο.

Αυτό σημαίνει πως ορίζουμε τιμές για τις παραμέτρους του μοντέλου.

Προσοχή: δεν δώσαμε εντολή ανάλογη με το model.fit(X,y) -- το συγκεκριμένο δίκτυο το επιλέξαμε εμείς, δεν το επέλεξε ο υπολογιστής ώστε να έχει μέγιστη ακρίβεια στα δεδομένα $(X,y)$.

In [ ]:
model = SimpleClassifier()

# Set weights and biases
model.fc.weight = nn.Parameter(torch.tensor([[1.0, 1.0], [-1.0, -1.0]]))
model.fc.bias = nn.Parameter(torch.tensor([-1.0, 1.0]))


**Τι κάνει το νευρωνικό:**

Με τις τιμές των παραμέτρων που έχουμε δώσει, έχουμε:
$$
{\rm model}(x_1,x_2) = \left(\begin{array}{c}z_1 \\ z_2\end{array} \right) = \left(\begin{array}{c} 1\cdot x_1 + 1 \cdot x_2 -1 \\ -1\cdot x_1 -1\cdot x_2 + 1 \end{array} \right)
$$
και μετά από το ``softmax``:
$$
\left(\begin{array}{c}p_1 \\ p_2\end{array} \right) = \left(\begin{array}{c}\frac{\exp(z_1)}{\exp(z_1) + \exp(z_2} \\ \frac{\exp(z_2)}{\exp(z_1) + \exp(z_2}\end{array} \right)
$$

## Πως τα πάει;

Να δούμε τώρα τι κάνει αυτό το νευρωνικό δίκτυο που ορίσαμε.
Θα διαλέξουμε τρία από τα δεδομένα που βλέπουμε απεικονισμένα στην εικόνα παραπάνω:

1. Το σημείο $(0.1,0.15)$ στην κάτω αριστερή γωνία.
2. Το σημείο $(0.8,0.8)$ στην πάνω δεξιά γωνία.
3. Το σημείο $(0.5,0.45)$ που είναι κάπου στην μέση.

**Ερώτηση**: Τι περιμένουμε σαν απαντήσεις από ένα "καλό" νευρωνικό δίκτυο; Εσείς τι θα δίνατε σαν απάντηση;

In [ ]:
# Select a few examples
points = torch.tensor([[0.1, 0.15],[0.8,0.8],[0.5,0.45]], dtype=torch.float32)

# Feed them through the network
outputs = model(points)

print("Outputs for selected points:\n", outputs)


**Άσκηση**
Προσπαθήστε να κάνετε τους υπολογισμούς μόνοι σας.

## Οπότε; Είναι Καλό;

Τι σημαίνουν οι αριθμοί που έδωσε για απάντηση;

1. Το σημείο $(0.1,0.15)$: [0.1824, 0.8176]
2. Το σημείο $(0.8,0.8)$: [0.7685, 0.2315]
3. Το σημείο $(0.5,0.45)$: [0.4750, 0.5250]


Βλέπουμε πως το νευρωνικό που διαλέξαμε -- δηλαδή το νευρωνικό δίκτυο που αντιστοιχεί στις παραμέτρους που διαλέξαμε, φαίνεται να συμφωνεί διαισθητικά με την εικόνα:

1. Το σημείο $(0.1,0.15)$ στην κάτω αριστερή γωνία: 81% κόκκινο
2. Το σημείο $(0.8,0.8)$ στην πάνω δεξιά γωνία: 77% μπλέ
3. Το σημείο $(0.5,0.45)$ που είναι κάπου στην μέση: Περίπου 50%-50%


### Ισοϋψείς καμπύλες
Τώρα θα οπτικοποιήσουμε τις "ισοϋψείς καμπύλες" για να καταλάβουμε καλύτερα την συμπεριφορά του νευρωνικού δικτύου.

Οι **ισοϋψείς καμπύλες** είναι τα σημεία στα οποία το μοντέλο μας αναθέτει την ίδια πιθανότητα να ανήκουν σε μία από τις δύο κατηγορίες.


In [ ]:
# Function to plot decision boundaries
def plot_decision_boundary(model, X, y):

    # make a fine mesh grid of points and label them with model
    x_min, x_max = X[:,0].min(), X[:,0].max()
    y_min, y_max = X[:,1].min(), X[:,1].max()
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    with torch.no_grad():
        probs = model(grid)
        Z = probs[:, 0] - probs[:, 1]
        Z = Z.reshape(xx.shape)

    plt.contourf(xx, yy, Z, cmap="RdBu", alpha=0.5)

    # Now plot the original points, along with their true label given in y
    # Extract positive and negative indices based on y
    positive_indices = np.where(y == 1)[0]
    negative_indices = np.where(y == 0)[0]

    # Split X into positive and negative components
    X_positive = X[positive_indices, :]
    X_negative = X[negative_indices, :]
    plt.scatter(X_positive[:,0], X_positive[:,1], marker='^', color='blue', label='Positive (Triangle)')
    plt.scatter(X_negative[:,0], X_negative[:,1], marker='o', color='red', label='Negative (Circle)')
    plt.title("Decision Region with Data Points")
    plt.show()




In [ ]:
X = np.vstack((X_positive,X_negative))
y = np.array([1]*20 + [0]*20)  # Correctly assigning labels to each class

plot_decision_boundary(model, X, y)

## Γραμμικές Καμπύλες

Παρατηρούμε πώς:
1. Όσο προχωράμε προς την κάτω αριστερά γωνία, τόσο πιο πολύ πιστεύει το μοντέλο μας πως τα σημεία είναι κόκκινα. Όσο προχωράμε προς τα πάνω δεξιά, τόσο το μοντέλο πιστεύει πως είναι μπλέ.

2. Οι ισοϋψείς καμπύλες είναι γραμμικές. Αυτό δεν είναι τυχαίο. Το δίκτυό μας είναι γραμμικό -- δεν περιέχει κανένα ReLU!





###**Άσκηση**

Να αλλάξετε τις τιμές των παραμέτρων του νευρωνικού δικτύου, και να δείτε πως αλλάζουν οι ισοϋψείς καμπύλες. Για παράδειγμα, δοκιμάστε τις τιμές:
```
α = 1, β = -1, γ = -1, δ = 1
c1 = -1, c2 = 1
```

Πιο δύσκολη άσκηση: Προσπαθήστε να προβλέψετε πως θα αλλάξουν οι καμπύλες!

### **Άσκηση**: πιο βαθύ δίκτυο

Στο προηγούμενο Notebook, δείξαμε πως χωρίς ReLU, το πιο βαθύ νευρωνικό δίκτυο παραμένει ισοδύναμο με ένα νευρωνικό δίκτυο με μόνο ένα επίπεδο.

Πιο συγκεκριμένα, χωρίς ReLU τα νευρωνικά δίκτυα ανεξαρτήτως του βάθους τους, παραμένουν **γραμμικά**, δηλαδή έχουν γραμμικές ισοϋψείς καμπύλες, και έτσι το μόνο που μπορούν να κάνουν είναι να περιγράψουν γραμμικούς διχωρισμούς του χώρου, που όπως θα δούμε και στο επόμενο παράδειγμα, δεν αρκούν.

1. **Πρώτο βήμα**: Φτιάξτε ένα νευρωνικό δίκτυο με τουλάχιστον δύο επίπεδα, αλλά χωρίς ReLU. Μπορείτε να χρησιμοποιήσετε κώδικα που έχουμε ήδη δεί. Το πρώτο επίπεδο πρέπει να έχει την μορφή
```
self.fc = nn.Linear(2, κ)
```
γιατί θέλουμε τα inputs να είναι δυσδιάστατα (αλλιώς δεν μπορούμε να τα απεικονίσουμε). Το τελευταίο επίπεδο πρέπει να έχει την μορφή
```
self.fc = nn.Linear(ν, 2)
```
δηλαδή να καταλήγει σε δύο output, για να ταιριάζει με τον κώδικα που δώσαμε παραπάνω που φτιάχνει τις ισοϋψείς καμπύλες. Επίσης, να διατηρήσετε την εντολή
```
return F.softmax(x, dim=1)
```
που υπολογίζει το τελικό ``softmax``.

2. **Δεύτερο βήμα**: χρησιμοποιήστε κώδικα όπως κάναμε και παραπάνω για να δώσουμε τιμές στις παραμέτρους του νευρωνικού. Θα χρησιμοποιήσετε εντολές όπως
```
model = YourClassifierName_Net()
model.fc.weight = nn.Parameter(torch.tensor([[1.0, 1.0], [-1.0, -1.0]]))
model.fc.bias = nn.Parameter(torch.tensor([-1.0, 1.0]))
```
Εκεί που βλέπετε ``model.fc`` θα χρησιμοποιείτε το όνομα του επιπέδου που δώσατε στο πρώτο ``def`` που ορίζει το νευρωνικό. Σκεφτείτε προσεχτικά πόσες παραμέτρους έχει το κάθε επίπεδο.

3. **Τρίτο βήμα**: Για να δείτε τις ισοϋψείς καμπύλες, μπορείτε να χρησιμοποιήσετε την εντολή
```
plot_decision_boundary(model, X, y)
```
οπού ``model`` είναι το μοντέλο σας.

# Δεύτερο Παράδειγμα

Τώρα θα δούμε ένα πιο περίπλοκο παράδειγμα που, όπως θα δούμε, απαιτεί βαθύτερο νευρωνικό δίκτυο για να πετύχει καλή (υψηλή) ακρίβεια.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Seed for reproducibility
np.random.seed(42)

# Generate positive examples centered around (1,1)
positive_x = 0.1 * np.random.randn(100, 2) + 1

# Generate negative examples in a ring-like structure
radius = 0.5 + 0.5 * np.random.rand(100, 1)
angle = 2 * np.pi * np.random.rand(100, 1)
negative_x = np.hstack((radius * np.cos(angle), radius * np.sin(angle))) + 1

# Plotting the data
plt.figure(figsize=(5, 5))
plt.scatter(positive_x[:, 0], positive_x[:, 1], marker='^', color='blue', label='Positive (Triangle)')
plt.scatter(negative_x[:, 0], negative_x[:, 1], marker='o', color='red', label='Negative (Circle)')
plt.title('Classification Dataset with Positive Center and Negative Ring')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True)
plt.show()


## Εδώ τι κάνουμε;

Θα μπορούσαμε να χρησιμοποιήσουμε κάποιο νευρωνικό δίκτυο από την παραπάνω οικογένεια για να ταξινομήσουμε αυτά τα δεδομένα;

Γιατί το πιστεύετε αυτό; Δηλαδή, γιατί ναι ή όχι; Πως θα θέλαμε να μοιάζουν οι ισοϋψείς καμπύλες για αυτό το παράδειγμα;